In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="2,3"

%load_ext autoreload
%autoreload 2

In [2]:
import ipdb
from Data import *
from address import *
from modelGen import *
from utils import cf_eval, cleanup_gpu
from models import CausalCondLatentCF, latentCFpp, PrototypeLatentCF
from models import CondLatentCF # Note: get the vanilla CondLatentCF
import numpy as np
import random 
from tensorflow import random as tf_random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from itertools import product
import argparse
import os
import pandas as pd


In [3]:
SEED = 23

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf_random.set_seed(SEED)

CF_METHODS = {  
    "LatentCFpp": latentCFpp,
    "Prototype": PrototypeLatentCF,
    "CausalCACTUS": CausalCondLatentCF,
    "CACTUS": CondLatentCF
}

In [4]:
# Experimentos que a realizar
EXP = [
    {
        "name": "LatentCF++",
        "data": "LAW",
        "classifier": "./exp/LAW_class/config.json",
        "AE": "./exp/LAW_AE/config.json",
        "CFmethod": "LatentCFpp",
        "context": [
            "male",
            "racetxt"
        ],
        "epochs": 100,
        "tol": 0.001,
        "target_prob": 0.5,
        "learning_rate": 0.1
    },
    {
        "name": "Prototype C",
        "data": "LAW",
        "classifier": "./exp/LAW_class/config.json",
        "AE": "./exp/LAW_AE/config.json",
        "CFmethod": "Prototype",
        "context": [
            "male",
            "racetxt"
        ],
        "epochs": 100,
        "kappa": 0.0,
        "c": 1.0,
        "c_steps": 1,
        "beta": 0.1,
        "gamma": 0.0,
        "theta": 100.0,
        "clip": [
            -1000,
            1000
        ],
        "feat_range": [
            -4,
            4
        ],
        "lr": 0.01
    },
    {
        "name": "Prototype D",
        "data": "LAW",
        "classifier": "./exp/LAW_class/config.json",
        "AE": "./exp/LAW_AE/config.json",
        "CFmethod": "Prototype",
        "context": [
            "male",
            "racetxt"
        ],
        "epochs": 100,
        "kappa": 0.0,
        "c": 1.0,
        "c_steps": 1,
        "beta": 0.1,
        "gamma": 100.0,
        "theta": 100.0,
        "clip": [
            -1000,
            1000
        ],
        "feat_range": [
            -4,
            4
        ],
        "lr": 0.01
    },
    {
        "name": "CACTUS",
        "data": "LAW",
        "classifier": "./exp/LAW_class/config.json",
        "AE": "./exp/LAW_CACTUS/config.json",
        "CFmethod": "CACTUS",
        "context": [
            "male",
            "racetxt"
        ],
        "epochs": 300,
        "target_prob": 0.5,
        # "learning_rate": 0.005,
        "learning_rate": 0.01,
        "power": 0.5,
        "alpha": 0.7,
        "gamma": 0.1,
        "beta": 0.01,
        # "dynamicAlpha": True
        "dynamicAlpha": False
    },
    {
        "name": "CausalCACTUS",
        "data": "LAW",
        "classifier": "./exp/LAW_class/config.json",
        "AE": "./exp/LAW_CACTUS/config.json",
        "CFmethod": "CausalCACTUS",
        "context": [
            "male",
            "racetxt"
        ],
        "epochs": 300,
        "target_prob": 0.5,
        # "learning_rate": 0.005,
        "learning_rate": 0.01,
        "power": 0.5,
        "alpha": 0.7,
        "gamma": 0.1,
        "beta": 0.01,
        "dynamicAlpha": True
        # "dynamicAlpha": False
    }
]


In [5]:
import re
from collections import defaultdict
# TODO: make a function to convert the causal paths for all the test samples
def build_causal_index_map(input_features, graph_str):
    """
    Returns:
        child_to_parents: dict {child_idx: [parent_idx, ...]}
        parent_to_children: dict {parent_idx: [child_idx, ...]}
        feat2idx: dict {feature_name: index}
    """

    # Feature name to index
    feat2idx = {f: i for i, f in enumerate(input_features)}

    # Extract edges using regex
    edges = re.findall(r'(\w+)\s*->\s*(\w+)', graph_str)

    child_to_parents = defaultdict(list)
    parent_to_children = defaultdict(list)

    for parent, child in edges:

        if parent not in feat2idx or child not in feat2idx:
            raise ValueError(f"Feature in graph not found in dataset: {parent} or {child}")

        p_idx = feat2idx[parent]
        c_idx = feat2idx[child]

        child_to_parents[c_idx].append(p_idx)
        parent_to_children[p_idx].append(c_idx)

    return dict(child_to_parents), dict(parent_to_children), feat2idx

# credit_rex5_constraints_min.dot
graph_str1 = """ 
strict digraph {
fulltime;
zgpa;
zfygpa;
faminc;
decile1b;
racetxt;
male;
ugpa;
decile3;
lsat;
tier;
fulltime -> faminc;
zgpa -> lsat;
zgpa -> ugpa;
zfygpa -> faminc;
decile1b -> lsat;
decile3 -> ugpa;
decile3 -> lsat;
decile3 -> faminc;
tier -> zgpa;
}
"""

# adult_rex6_constraints_mod.dot
graph_str2 = """
strict digraph {
fulltime;
zgpa;
zfygpa;
faminc;
decile1b;
racetxt;
male;
ugpa;
decile3;
lsat;
tier;
fulltime -> faminc;
ugpa -> decile3;
lsat -> decile1b;
}
"""

# # CF generation
# child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
#     data.features_lbls,
#     # graph_str
#     graph_str1
# )
# print("Feature → index:")
# print(feat2idx)
# print("\nChild → Parents (index form):")
# print(child_to_parents_dict)

In [6]:
# data.features_lbls

## Graph 1 (Minimum priors)

In [7]:
N = 50 
N_BOOTSTRAP = 5
# SEED_list = [23] * N_BOOTSTRAP
SEED_list = [12, 23, 34, 45, 56]

# Number of samples to compute the metrics for CF evaluation
METRICS = []
for i, exp in enumerate(EXP):
    print("\n" * 2)
    print("*" * 200)
    print(f"Running exp: {exp['name']} ({i}/{len(EXP)})")
    print("*" * 200)
    print("\n" * 2)
    
    # Reading the data
    CLASS_CONFIG_PATH = exp["classifier"]
    AE_CONFIG_PATH = exp["AE"]

    class_config = get_exp_config(CLASS_CONFIG_PATH)
    print("Reading data")
    data = getData(class_config)

    # Getting classifier
    classifier = modelGen(class_config["type"], data, params=class_config, debug=True)
    classifier.load()

    # Getting AE-based Model
    AE_config = get_exp_config(AE_CONFIG_PATH)
    aeModel = modelGen(AE_config["type"], data, params=AE_config, debug=True)
    aeModel.load()

    # CF generation
    child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
        data.features_lbls,
        # TODO: change the string below for graph_str1 and graph_str2
        graph_str1
    )
    print("Feature → index:")
    print(feat2idx)
    print("\nChild → Parents (index form):")
    print(child_to_parents_dict)
    
    if exp["name"] == "CausalCACTUS":
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train,
            x_min=data.X_train.min(axis=0),
            x_max=data.X_train.max(axis=0),
            causal_index_map=child_to_parents_dict
        )
    else:
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train
        )

    for trial in range(N_BOOTSTRAP):

        # --- Sampling test data ---
        SEED = SEED_list[trial]
        os.environ['PYTHONHASHSEED'] = str(SEED)
        random.seed(SEED)
        np.random.seed(SEED)
        tf_random.set_seed(SEED)
        rand_idx = np.random.choice(len(data.X_test), N, replace=True)
        
        # previous name: x
        X_test = data.X_test[rand_idx, ...]
        # previous name: contest
        context_test = data.context_test[exp['context']].values
        context_test = context_test[rand_idx, ...]

        context_training = data.context_train[exp['context']].values

        # previous name: y
        y_test = np.argmax(data.y_test[rand_idx, ...], axis=1)
        y_original_logits = classifier.predict(X_test)
        # previous name: y_
        y_original_labels = np.argmax(y_original_logits, axis=1)

        # --- CF generation ---
        # previous names: 
        # cf_x, cf_y_, x, __
        X_cf, y_cf_labels, X_internal, y_internal = CF_method.transform(
            X_test,
            y_original_labels,
            target_context=context_test,
            verbose=0
        )              

        # TODO: One-hot encoding function for datasets with catergorical features
        cat_idx = [8] #     "tier"
        num_idx = [0, 1, 2, 3, 4, 5, 6, 7]
        X_cf[:, cat_idx] = np.around(X_cf[:, cat_idx])
        
        # --- CF evaluation ---
        cf_scores, cf_scores_labels = cf_eval(
            data.X_train, # previous name: data.X_train
            context_training, # previous name: context_training,
            X_test, # previous name: x,
            X_cf, # previous name: cf_x, 
            y_original_labels, # previous name: y_,
            context_test, # previous name: context,
            y_cf_labels, # previous name: cf_y_
            data.scaler_inverse_transform(X_test),
            data.scaler_inverse_transform(X_cf),
            child_to_parents_dict, 
            cat_idx,
            num_idx
        )

        for i_metric, label in enumerate(cf_scores_labels):
            METRICS.append([
                exp["name"],
                exp["data"],
                exp["CFmethod"],
                label,
                cf_scores[i_metric]
            ])

    # Cleaning GPU models
    cleanup_gpu()

# --- Metrics aggregation ---
df_metrics = pd.DataFrame(
    data=METRICS,
    columns=["Model", "Dataset", "CFMethod", "Metric", "Value"]
)
df_metrics = df_metrics.round(3)
print(df_metrics)




********************************************************************************************************************************************************************************************************
Running exp: LatentCF++ (0/5)
********************************************************************************************************************************************************************************************************



Reading data
male
1    10550
0     8142
Name: count, dtype: int64
racetxt
1    17491
0     1201
Name: count, dtype: int64




----------------------------------------------------------------------------------------------------
---- DATA SUMMARY ----
Total samples: 3672
Positive class: 1836.0
Context 'sex=Male': 1992
Context 'race=White': 3138
----------------------------------------------------------------------------------------------------




Building model


/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_Law.py:100: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["male"]))
/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_Law.py:101: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["racetxt"]))


Building model: DNN [class]
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 main_input (InputLayer)     [(None, 9)]               0         
                                                                 
 dense (Dense)               (None, 64)                640       
                                                                 
 batch_normalization (BatchN  (None, 64)               256       
 ormalization)                                                   
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 128)               8320      
                                                                 
 batch_normalization_1 (Batc  (None, 128)              512       
 hNormalization)                 

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


restoring weights of model AE: LAW_AE
Feature → index:
{'lsat': 0, 'ugpa': 1, 'zfygpa': 2, 'zgpa': 3, 'faminc': 4, 'decile1b': 5, 'decile3': 6, 'fulltime': 7, 'tier': 8}

Child → Parents (index form):
{4: [7, 2, 6], 0: [3, 5, 6], 1: [3, 6], 3: [8]}
Building model: latentCF [CF]
Model: "model_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 z_input (InputLayer)        [(None, 16)]              0         
                                                                 
 model_2 (Functional)        (None, 9)                 4665      
                                                                 
 model (Functional)          (None, 2)                 35394     
                                                                 
Total params: 40,059
Trainable params: 39,291
Non-trainable params: 768
_________________________________________________________________
2/2 [==============================] -

...

Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 main_input (InputLayer)        [(None, 9)]          0           []                               
                                                                                                  
 model_1 (Functional)           (None, 4)            1508        ['main_input[0][0]',             
                                                                  'main_input[0][0]']             
                                                                                                  
 model_2 (Functional)           (None, 9)            1513        ['model_1[0][0]']                
                                                                                                  
 model_3 (Functional)           (None, 2)            10          ['model_1[1][0]']          

/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_Law.py:100: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["male"]))
/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_Law.py:101: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["racetxt"]))


 dense_2 (Dense)             (None, 128)               16512     
                                                                 
 batch_normalization_2 (Batc  (None, 128)              512       
 hNormalization)                                                 
                                                                 
 dropout_2 (Dropout)         (None, 128)               0         
                                                                 
 dense_3 (Dense)             (None, 64)                8256      
                                                                 
 batch_normalization_3 (Batc  (None, 64)               256       
 hNormalization)                                                 
                                                                 
 dropout_3 (Dropout)         (None, 64)                0         
                                                                 
 dense_4 (Dense)             (None, 2)                 130       
          

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 main_input (InputLayer)        [(None, 9)]          0           []                               
                                                                                                  
 model_1 (Functional)           (None, 4)            1508        ['main_input[0][0]',             
                                                                  'main_input[0][0]']             
                                                                                                  
 model_2 (Functional)           (None, 9)            1513        ['model_1[0][0]']                
                                                                                                  
 model_3 (Functional)           (None, 2)            10          ['model_1[1][0]']          

In [8]:
child_to_parents_dict

{4: [7, 2, 6], 0: [3, 5, 6], 1: [3, 6], 3: [8]}

In [9]:
# Pivoting the dataframe
df_metrics_pivot = df_metrics.pivot_table(
    index=["Dataset", "Model"],
    columns="Metric",
    values="Value",
    aggfunc=['mean', 'std']
)

desired_order = [
    "validity",
    "lof_context",
    "compactness",
    "n_proximity",
    "causal_validity_hard",
    "causal_validity_soft",
    "causal_compact_val",
    "inlier_fraction"
]
df_metrics_pivot.columns = df_metrics_pivot.columns.swaplevel(0, 1)
df_metrics_pivot = df_metrics_pivot[desired_order]

df_metrics_pivot.rename(
    columns={
        "validity": "Validity",
        "lof_context": "LOF",
        "compactness": "Compactness",
        "n_proximity": "Proximity",
        "causal_validity_hard": "Causal Validity (Hard)",
        "causal_validity_soft": "Causal Validity (Soft)",
        "causal_compact_val": "Causal Compact Validity",
        "inlier_fraction": "inlier_fraction"
    },
    inplace=True
)

df_metrics_pivot.columns = df_metrics_pivot.columns.swaplevel(0, 1)

# df_formatted = df_metrics_pivot.apply(
#     lambda x: x["mean"].map("${:2.2f}".format)
#     + " \\pm "
#     + x["std"].map("{:2.2f} $".format),
#     axis=1
# )
# df_formatted

df_formatted = df_metrics_pivot.apply(
    lambda x: (
        x["mean"].round(2).astype(str)
        + " ± "
        + x["std"].round(2).astype(str)
    ),
    axis=1
)
df_formatted

Metric                   Validity          LOF  Compactness    Proximity  \
Dataset Model                                                              
LAW     CACTUS        0.71 ± 0.05  1.06 ± 0.02  0.19 ± 0.05  0.52 ± 0.04   
        CausalCACTUS   0.8 ± 0.05  1.04 ± 0.01  0.29 ± 0.04  0.38 ± 0.03   
        LatentCF++     1.0 ± 0.01   1.1 ± 0.02  0.13 ± 0.01  0.51 ± 0.04   
        Prototype C     1.0 ± 0.0  2.65 ± 0.07  0.03 ± 0.01   2.3 ± 0.05   
        Prototype D     1.0 ± 0.0  2.01 ± 0.07  0.03 ± 0.01  1.84 ± 0.07   

Metric               Causal Validity (Hard) Causal Validity (Soft)  \
Dataset Model                                                        
LAW     CACTUS                  0.51 ± 0.07            0.88 ± 0.02   
        CausalCACTUS            0.48 ± 0.07            0.87 ± 0.02   
        LatentCF++              0.21 ± 0.06             0.8 ± 0.02   
        Prototype C             0.82 ± 0.05            0.95 ± 0.01   
        Prototype D              0.8 ± 0.08            0.95 ± 0.02   

Metric               Causal Compact Validity inlier_fraction  
Dataset Model                                                 
LAW     CACTUS                   0.26 ± 0.05       1.0 ± 0.0  
        CausalCACTUS             0.33 ± 0.04       1.0 ± 0.0  
        LatentCF++                0.17 ± 0.0     0.98 ± 0.04  
        Prototype C               0.08 ± 0.0     0.04 ± 0.02  
        Prototype D              0.09 ± 0.01     0.14 ± 0.05

In [10]:
print(data.scaler_inverse_transform(X_test))
print(data.scaler_inverse_transform(X_cf))

[[ 2.70e+01  2.90e+00 -3.60e-01 -1.09e+00  4.00e+00  4.00e+00  2.00e+00
   1.00e+00  1.00e+00]
 [ 3.60e+01  3.60e+00  5.90e-01  0.00e+00  3.00e+00  7.00e+00  5.00e+00
   1.00e+00  5.00e+00]
 [ 4.10e+01  2.50e+00  1.00e-01 -9.50e-01  3.00e+00  6.00e+00  2.00e+00
   2.00e+00  2.00e+00]
 [ 3.70e+01  3.90e+00  1.04e+00  2.12e+00  4.00e+00  9.00e+00  1.00e+01
   1.00e+00  1.00e+00]
 [ 4.35e+01  3.00e+00  1.36e+00  1.22e+00  3.00e+00  1.00e+01  9.00e+00
   1.00e+00  4.00e+00]
 [ 3.90e+01  3.30e+00 -1.30e-01 -1.10e+00  4.00e+00  5.00e+00  2.00e+00
   1.00e+00  2.00e+00]
 [ 3.25e+01  3.50e+00  1.40e-01 -1.60e-01  3.00e+00  6.00e+00  5.00e+00
   1.00e+00  2.00e+00]
 [ 3.70e+01  3.20e+00 -5.10e-01 -9.90e-01  3.00e+00  3.00e+00  2.00e+00
   1.00e+00  3.00e+00]
 [ 3.70e+01  2.70e+00 -6.30e-01 -1.43e+00  4.00e+00  3.00e+00  1.00e+00
   1.00e+00  1.00e+00]
 [ 3.60e+01  2.50e+00  1.30e-01  1.80e-01  4.00e+00  6.00e+00  6.00e+00
   1.00e+00  1.00e+00]
 [ 4.70e+01  3.90e+00  2.20e-01 -2.00e-02  3.00e+0

In [11]:
# TODO: One-hot encode categorical features
# cat_idx = [1, 2, 6] # "workclass", "occupation", "maritalstatus"
# num_idx = [0, 3, 4, 5]
# X_cf[:, cat_idx] = np.around(X_cf[:, cat_idx])

# --- CF evaluation ---
cf_scores, cf_scores_labels = cf_eval(
    data.X_train, # previous name: data.X_train
    context_training, # previous name: context_training,
    X_test, # previous name: x,
    X_cf, # previous name: cf_x, 
    y_original_labels, # previous name: y_,
    context_test, # previous name: context,
    y_cf_labels, # previous name: cf_y_
    data.scaler_inverse_transform(X_test),
    data.scaler_inverse_transform(X_cf),
    child_to_parents_dict,
    cat_idx,
    num_idx
)

print(cf_scores, cf_scores_labels)

[1 0 1 0 0 1 0 0 1 1 0 1 1 0 0 0 1 1 1 0 1 1 1 0 1 0 1 0 1 1 0 1 1 1 1 0 1
 0 1 1 0 1 0 0 1 0 0 1 0 0]
[1 0 1 0 1 1 0 0 1 1 0 1 0 0 0 0 1 1 1 0 1 0 1 0 1 0 1 0 1 1 0 1 1 1 1 0 0
 0 0 1 0 1 0 0 1 0 0 0 0 0]
0.88
---- DEBUG Context-LOF score ----
Context: [0, 0]
Subset size: 277
Unique rows: 277
After np.unique(): subset size: 277
cf_samples_ shape: (3, 14)
X_ shape: (277, 14)
lof_cf min/max: 1.0038540023649731 1.0702465089471858
lof_train min/max: 0.9471415053747453 1.546608176758252
mean train LOF: 1.054292255128558
---- DEBUG Context-LOF score ----
Context: [0, 1]
Subset size: 1090
Unique rows: 1090
After np.unique(): subset size: 1090
cf_samples_ shape: (12, 14)
X_ shape: (1090, 14)
lof_cf min/max: 0.9510664947867516 1.1149647747792517
lof_train min/max: 0.9597768787247467 1.9180628303323088
mean train LOF: 1.058482696208251
---- DEBUG Context-LOF score ----
Context: [1, 0]
Subset size: 165
Unique rows: 165
After np.unique(): subset size: 165
cf_samples_ shape: (4, 14)
X_ shape: (165

## Graph 2 (Moderate priors)

In [12]:
N = 50 
N_BOOTSTRAP = 5
# SEED_list = [23] * N_BOOTSTRAP
SEED_list = [11, 22, 33, 44, 55]

# Number of samples to compute the metrics for CF evaluation
METRICS = []
for i, exp in enumerate(EXP):
    print("\n" * 2)
    print("*" * 200)
    print(f"Running exp: {exp['name']} ({i}/{len(EXP)})")
    print("*" * 200)
    print("\n" * 2)
    
    # Reading the data
    CLASS_CONFIG_PATH = exp["classifier"]
    AE_CONFIG_PATH = exp["AE"]

    class_config = get_exp_config(CLASS_CONFIG_PATH)
    print("Reading data")
    data = getData(class_config)

    # Getting classifier
    classifier = modelGen(class_config["type"], data, params=class_config, debug=True)
    classifier.load()

    # Getting AE-based Model
    AE_config = get_exp_config(AE_CONFIG_PATH)
    aeModel = modelGen(AE_config["type"], data, params=AE_config, debug=True)
    aeModel.load()

    # CF generation
    child_to_parents_dict, parent_to_children_dict, feat2idx = build_causal_index_map(
        data.features_lbls,
        # TODO: change the string below for graph_str1 and graph_str2
        graph_str2
    )
    print("Feature → index:")
    print(feat2idx)
    print("\nChild → Parents (index form):")
    print(child_to_parents_dict)
    
    if exp["name"] == "CausalCACTUS":
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train,
            x_min=data.X_train.min(axis=0),
            x_max=data.X_train.max(axis=0),
            causal_index_map=child_to_parents_dict
        )
    else:
        CF_method = CF_METHODS[exp["CFmethod"]](
            classifier=classifier,
            gen=aeModel,
            params=exp,
            x=data.X_train,
            y=data.y_train
        )

    for trial in range(N_BOOTSTRAP):

        # --- Sampling test data ---
        SEED = SEED_list[trial]
        os.environ['PYTHONHASHSEED'] = str(SEED)
        random.seed(SEED)
        np.random.seed(SEED)
        tf_random.set_seed(SEED)
        rand_idx = np.random.choice(len(data.X_test), N, replace=True)
        
        # previous name: x
        X_test = data.X_test[rand_idx, ...]
        # previous name: contest
        context_test = data.context_test[exp['context']].values
        context_test = context_test[rand_idx, ...]

        context_training = data.context_train[exp['context']].values

        # previous name: y
        y_test = np.argmax(data.y_test[rand_idx, ...], axis=1)
        y_original_logits = classifier.predict(X_test)
        # previous name: y_
        y_original_labels = np.argmax(y_original_logits, axis=1)

        # --- CF generation ---
        # previous names: 
        # cf_x, cf_y_, x, __
        X_cf, y_cf_labels, X_internal, y_internal = CF_method.transform(
            X_test,
            y_original_labels,
            target_context=context_test,
            verbose=0
        )              

        # TODO: One-hot encoding function for datasets with catergorical features
        cat_idx = [8] #     "tier"
        num_idx = [0, 1, 2, 3, 4, 5, 6, 7]
        X_cf[:, cat_idx] = np.around(X_cf[:, cat_idx])
        
        # --- CF evaluation ---
        cf_scores, cf_scores_labels = cf_eval(
            data.X_train, # previous name: data.X_train
            context_training, # previous name: context_training,
            X_test, # previous name: x,
            X_cf, # previous name: cf_x, 
            y_original_labels, # previous name: y_,
            context_test, # previous name: context,
            y_cf_labels, # previous name: cf_y_
            data.scaler_inverse_transform(X_test),
            data.scaler_inverse_transform(X_cf),
            child_to_parents_dict, 
            cat_idx,
            num_idx
        )

        for i_metric, label in enumerate(cf_scores_labels):
            METRICS.append([
                exp["name"],
                exp["data"],
                exp["CFmethod"],
                label,
                cf_scores[i_metric]
            ])

    # Cleaning GPU models
    cleanup_gpu()

# --- Metrics aggregation ---
df_metrics = pd.DataFrame(
    data=METRICS,
    columns=["Model", "Dataset", "CFMethod", "Metric", "Value"]
)
df_metrics = df_metrics.round(3)
print(df_metrics)




********************************************************************************************************************************************************************************************************
Running exp: LatentCF++ (0/5)
********************************************************************************************************************************************************************************************************



Reading data
male
1    10550
0     8142
Name: count, dtype: int64
racetxt
1    17491
0     1201
Name: count, dtype: int64




----------------------------------------------------------------------------------------------------
---- DATA SUMMARY ----
Total samples: 3672
Positive class: 1836.0
Context 'sex=Male': 1992
Context 'race=White': 3138
----------------------------------------------------------------------------------------------------




Building model
Building model: DNN [class]
Model: "model"
______________________________________________________

/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_Law.py:100: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["male"]))
/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_Law.py:101: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["racetxt"]))


 hNormalization)                                                 
                                                                 
 dropout_2 (Dropout)         (None, 128)               0         
                                                                 
 dense_3 (Dense)             (None, 64)                8256      
                                                                 
 batch_normalization_3 (Batc  (None, 64)               256       
 hNormalization)                                                 
                                                                 
 dropout_3 (Dropout)         (None, 64)                0         
                                                                 
 dense_4 (Dense)             (None, 2)                 130       
                                                                 
Total params: 35,394
Trainable params: 34,626
Non-trainable params: 768
_________________________________________________________________
rest

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


Feature → index:
{'lsat': 0, 'ugpa': 1, 'zfygpa': 2, 'zgpa': 3, 'faminc': 4, 'decile1b': 5, 'decile3': 6, 'fulltime': 7, 'tier': 8}

Child → Parents (index form):
{4: [7], 6: [1], 5: [0]}
Building model: latentCF [CF]
Model: "model_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 z_input (InputLayer)        [(None, 16)]              0         
                                                                 
 model_2 (Functional)        (None, 9)                 4665      
                                                                 
 model (Functional)          (None, 2)                 35394     
                                                                 
Total params: 40,059
Trainable params: 39,291
Non-trainable params: 768
_________________________________________________________________
2/2 [==============================] - 0s 3ms/step
Generating CF 0/50
Generating CF 1/50
Generating

...

Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 main_input (InputLayer)        [(None, 9)]          0           []                               
                                                                                                  
 model_1 (Functional)           (None, 4)            1508        ['main_input[0][0]',             
                                                                  'main_input[0][0]']             
                                                                                                  
 model_2 (Functional)           (None, 9)            1513        ['model_1[0][0]']                
                                                                                                  
 model_3 (Functional)           (None, 2)            10          ['model_1[1][0]']          

/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_Law.py:100: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["male"]))
/home/zhwang/Projects/CondLatentCF_git/Data/DataReader_Law.py:101: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print(pd.value_counts(self.context["racetxt"]))


                                                                 
 dropout_1 (Dropout)         (None, 128)               0         
                                                                 
 dense_2 (Dense)             (None, 128)               16512     
                                                                 
 batch_normalization_2 (Batc  (None, 128)              512       
 hNormalization)                                                 
                                                                 
 dropout_2 (Dropout)         (None, 128)               0         
                                                                 
 dense_3 (Dense)             (None, 64)                8256      
                                                                 
 batch_normalization_3 (Batc  (None, 64)               256       
 hNormalization)                                                 
                                                                 
 dropout_3

/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer VarianceScaling is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(
/home/zhwang/miniconda3/envs/py39/lib/python3.9/site-packages/keras/initializers/initializers_v2.py:120: UserWarning: The initializer RandomNormal is unseeded and being called multiple times, which will return identical values  each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initalizer instance more than once.
  warnings.warn(


Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 main_input (InputLayer)        [(None, 9)]          0           []                               
                                                                                                  
 model_1 (Functional)           (None, 4)            1508        ['main_input[0][0]',             
                                                                  'main_input[0][0]']             
                                                                                                  
 model_2 (Functional)           (None, 9)            1513        ['model_1[0][0]']                
                                                                                                  
 model_3 (Functional)           (None, 2)            10          ['model_1[1][0]']          

In [13]:
child_to_parents_dict

{4: [7], 6: [1], 5: [0]}

In [14]:
# Pivoting the dataframe
df_metrics_pivot = df_metrics.pivot_table(
    index=["Dataset", "Model"],
    columns="Metric",
    values="Value",
    aggfunc=['mean', 'std']
)

desired_order = [
    "validity",
    "lof_context",
    "compactness",
    "n_proximity",
    "causal_validity_hard",
    "causal_validity_soft",
    "causal_compact_val",
    "inlier_fraction"
]
df_metrics_pivot.columns = df_metrics_pivot.columns.swaplevel(0, 1)
df_metrics_pivot = df_metrics_pivot[desired_order]

df_metrics_pivot.rename(
    columns={
        "validity": "Validity",
        "lof_context": "LOF",
        "compactness": "Compactness",
        "n_proximity": "Proximity",
        "causal_validity_hard": "Causal Validity (Hard)",
        "causal_validity_soft": "Causal Validity (Soft)",
        "causal_compact_val": "Causal Compact Validity",
        "inlier_fraction": "inlier_fraction"
    },
    inplace=True
)

df_metrics_pivot.columns = df_metrics_pivot.columns.swaplevel(0, 1)

# df_formatted = df_metrics_pivot.apply(
#     lambda x: x["mean"].map("${:2.2f}".format)
#     + " \\pm "
#     + x["std"].map("{:2.2f} $".format),
#     axis=1
# )
# df_formatted

df_formatted = df_metrics_pivot.apply(
    lambda x: (
        x["mean"].round(2).astype(str)
        + " ± "
        + x["std"].round(2).astype(str)
    ),
    axis=1
)
df_formatted

Metric                   Validity          LOF  Compactness    Proximity  \
Dataset Model                                                              
LAW     CACTUS        0.68 ± 0.08  1.06 ± 0.01  0.16 ± 0.02  0.53 ± 0.04   
        CausalCACTUS  0.83 ± 0.06  1.04 ± 0.01  0.27 ± 0.06  0.39 ± 0.04   
        LatentCF++    0.99 ± 0.01  1.11 ± 0.02  0.12 ± 0.03  0.53 ± 0.07   
        Prototype C     1.0 ± 0.0  2.54 ± 0.09  0.02 ± 0.01  2.26 ± 0.05   
        Prototype D     1.0 ± 0.0  2.01 ± 0.06  0.04 ± 0.01  1.81 ± 0.06   

Metric               Causal Validity (Hard) Causal Validity (Soft)  \
Dataset Model                                                        
LAW     CACTUS                   0.1 ± 0.04            0.69 ± 0.01   
        CausalCACTUS            0.22 ± 0.06            0.73 ± 0.03   
        LatentCF++                0.0 ± 0.0             0.67 ± 0.0   
        Prototype C             0.44 ± 0.02            0.81 ± 0.01   
        Prototype D             0.37 ± 0.05            0.79 ± 0.02   

Metric               Causal Compact Validity inlier_fraction  
Dataset Model                                                 
LAW     CACTUS                    0.2 ± 0.02      1.0 ± 0.01  
        CausalCACTUS             0.28 ± 0.06      1.0 ± 0.01  
        LatentCF++               0.14 ± 0.01     0.98 ± 0.01  
        Prototype C               0.06 ± 0.0     0.02 ± 0.02  
        Prototype D              0.07 ± 0.01     0.12 ± 0.02